# BÀI THỰC HÀNH CHƯƠNG 2 (PHẦN 2): CANNY EDGE DETECTOR
# PHẦN I — LÝ THUYẾT & CÂU HỎI MỞ RỘNG

---

# I. LÝ THUYẾT

## 1.1. Tìm hiểu về các bước của thuật toán Canny

### a) Giải thích chi tiết từng bước

Thuật toán Canny Edge Detector (do John F. Canny đề xuất năm 1986) là một thuật toán phát hiện cạnh đa bước, được coi là **chuẩn vàng (gold standard)** trong lĩnh vực phát hiện cạnh. Thuật toán gồm **5 bước chính**:

---

**Bước 1: Giảm nhiễu bằng bộ lọc Gaussian (Gaussian Smoothing)**

- **Mục đích:** Loại bỏ nhiễu trong ảnh vì nhiễu có thể tạo ra các cạnh giả (false edges).
- **Phương pháp:** Áp dụng bộ lọc Gaussian (convolution) lên ảnh gốc.
- **Công thức kernel Gaussian 2D:**

$$G(x, y) = \frac{1}{2\pi\sigma^2} e^{-\frac{x^2 + y^2}{2\sigma^2}}$$

- Trong đó $\sigma$ (sigma) là độ lệch chuẩn, quyết định mức độ làm mịn.
- Kích thước kernel thường là $5 \times 5$.
- $\sigma$ càng lớn → ảnh càng mịn → mất nhiều chi tiết nhỏ nhưng loại bỏ nhiễu tốt hơn.

---

**Bước 2: Tính toán Gradient (Gradient Computation)**

- **Mục đích:** Xác định cường độ và hướng của gradient tại mỗi pixel.
- **Phương pháp:** Sử dụng toán tử Sobel theo 2 hướng $x$ và $y$:

$$G_x = \begin{bmatrix} -1 & 0 & +1 \\ -2 & 0 & +2 \\ -1 & 0 & +1 \end{bmatrix} * I, \quad G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ +1 & +2 & +1 \end{bmatrix} * I$$

- **Cường độ gradient (magnitude):**

$$|G| = \sqrt{G_x^2 + G_y^2}$$

- **Hướng gradient (direction):**

$$\theta = \arctan\left(\frac{G_y}{G_x}\right)$$

- Hướng gradient được làm tròn về 4 hướng chính: **0°, 45°, 90°, 135°**.

---

**Bước 3: Triệt tiêu phi cực đại (Non-Maximum Suppression - NMS)**

- **Mục đích:** Làm mỏng các cạnh, chỉ giữ lại các pixel có cường độ gradient cực đại cục bộ.
- **Phương pháp:**
  - Tại mỗi pixel, so sánh cường độ gradient của nó với 2 pixel lân cận **theo hướng gradient**.
  - Nếu pixel hiện tại có cường độ lớn nhất → **giữ lại**.
  - Nếu không → **loại bỏ (đặt bằng 0)**.
- **Kết quả:** Các cạnh trở nên mỏng (thin edges), chỉ còn 1 pixel chiều rộng.

---

**Bước 4: Ngưỡng kép (Double Thresholding / Hysteresis Thresholding)**

- **Mục đích:** Phân loại các pixel cạnh thành 3 nhóm dựa trên 2 ngưỡng.
- **Hai ngưỡng:** Ngưỡng cao ($T_{high}$) và ngưỡng thấp ($T_{low}$).
- **Phân loại:**
  - $|G| \geq T_{high}$: **Cạnh mạnh (Strong edge)** → chắc chắn là cạnh.
  - $T_{low} \leq |G| < T_{high}$: **Cạnh yếu (Weak edge)** → có thể là cạnh.
  - $|G| < T_{low}$: **Không phải cạnh** → loại bỏ.
- **Tỷ lệ khuyến nghị:** $T_{high} : T_{low} = 2:1$ hoặc $3:1$.

---

**Bước 5: Theo dõi cạnh bằng trễ (Edge Tracking by Hysteresis)**

- **Mục đích:** Quyết định các cạnh yếu có phải là cạnh thật hay không.
- **Phương pháp:**
  - Nếu một cạnh yếu **kết nối** (liền kề) với ít nhất một cạnh mạnh → **giữ lại** (coi là cạnh thật).
  - Nếu một cạnh yếu **không kết nối** với cạnh mạnh nào → **loại bỏ** (coi là nhiễu).
  - Sử dụng thuật toán duyệt đồ thị (BFS/DFS) trên vùng lân cận $8$ pixel.
- **Kết quả cuối cùng:** Bản đồ cạnh nhị phân (binary edge map) chất lượng cao.

### b) So sánh Canny với các thuật toán Sobel và Laplacian

| Tiêu chí | **Sobel** | **Laplacian** | **Canny** |
|:---|:---|:---|:---|
| **Nguyên lý** | Đạo hàm bậc 1 (gradient) | Đạo hàm bậc 2 (Laplacian) | Đa bước: Gaussian + Gradient + NMS + Hysteresis |
| **Loại cạnh** | Cạnh dày (thick) | Cạnh kép (double edges) tại zero-crossing | Cạnh mỏng, sắc nét (1 pixel) |
| **Xử lý nhiễu** | Nhạy với nhiễu vừa phải | **Rất nhạy** với nhiễu | **Tốt nhất** nhờ Gaussian smoothing |
| **Kết quả** | Gradient magnitude map | Zero-crossing map | Binary edge map chính xác |
| **Tham số** | Kích thước kernel | Kích thước kernel | $\sigma$, $T_{low}$, $T_{high}$ |
| **Độ phức tạp** | Thấp | Thấp | **Cao hơn** (nhiều bước xử lý) |
| **Chất lượng** | Trung bình | Thấp (do nhạy nhiễu) | **Cao nhất** |
| **Ứng dụng** | Tiền xử lý, feature extraction | Phát hiện blob, sharpening | Phát hiện cạnh chính xác, thị giác máy |

## 1.2. Các tham số của thuật toán và ảnh hưởng của chúng

### a) Sigma ($\sigma$) trong Gaussian Filter

- **$\sigma$ nhỏ (0.5 – 1.0):**
  - Làm mịn ít → giữ lại nhiều chi tiết và cạnh nhỏ.
  - Nhưng cũng giữ lại nhiều nhiễu → phát hiện nhiều cạnh giả.
  
- **$\sigma$ lớn (2.0 – 5.0):**
  - Làm mịn mạnh → loại bỏ nhiễu hiệu quả.
  - Nhưng cũng làm mờ các cạnh yếu và chi tiết nhỏ → bỏ sót cạnh.
  
- **Kết luận:** Giá trị $\sigma = 1.0 – 1.4$ thường cho kết quả tốt trong thực tế.

### b) Ngưỡng thấp ($T_{low}$) và ngưỡng cao ($T_{high}$)

- **Ngưỡng cao ($T_{high}$):**
  - Cao → chỉ phát hiện cạnh mạnh, ít cạnh nhưng chính xác.
  - Thấp → nhiều cạnh hơn, nhưng có thể bao gồm cạnh giả.
  
- **Ngưỡng thấp ($T_{low}$):**
  - Cao → cạnh bị đứt đoạn.
  - Thấp → cạnh liền mạch hơn nhưng nhiều nhiễu.
  
- **Mối quan hệ:** Tỷ lệ $T_{high} / T_{low}$ thường từ **2:1** đến **3:1**.
  - Ví dụ: $T_{low} = 50$, $T_{high} = 150$ (tỷ lệ 3:1).

## 1.3. Ưu điểm, nhược điểm và ứng dụng thực tế

### a) So sánh với các thuật toán khác

| Tiêu chí | **Canny** | **Sobel** | **Laplacian** | **Prewitt** |
|:---|:---|:---|:---|:---|
| **Độ chính xác** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ |
| **Tốc độ** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Xử lý nhiễu** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐ | ⭐⭐⭐ |
| **Cạnh mỏng** | ✅ (NMS) | ❌ | ❌ | ❌ |
| **Cạnh liên tục** | ✅ (Hysteresis) | ❌ | ❌ | ❌ |

### b) Lĩnh vực sử dụng phổ biến

- **Thị giác máy tính:** Phát hiện đối tượng, nhận dạng khuôn mặt, ADAS.
- **Y tế:** Phát hiện biên khối u trong MRI/CT.
- **Robot & tự động hóa:** Điều hướng robot, kiểm tra chất lượng.
- **Xử lý tài liệu:** OCR, phát hiện vùng văn bản.
- **Viễn thám:** Phân tích ảnh vệ tinh.

### c) Ví dụ cụ thể

1. **Xe tự lái:** Phát hiện làn đường (lane detection) kết hợp Hough Transform.
2. **Chẩn đoán y khoa:** Phát hiện đường viền khối u trong X-quang/MRI.
3. **Kiểm tra công nghiệp:** Phát hiện vết nứt, khuyết tật bề mặt.
4. **Nhận dạng biển số xe:** Tách biên ký tự cho hệ thống ANPR.

---

**Ưu điểm:**
- Cạnh chính xác, mỏng, liên tục.
- Lọc nhiễu tốt nhờ Gaussian.
- Điều chỉnh linh hoạt qua tham số.

**Nhược điểm:**
- Chậm hơn Sobel/Laplacian.
- Cần tinh chỉnh tham số cho từng loại ảnh.
- Không tốt với soft edges (gradient thay đổi chậm).
- Phụ thuộc vào chất lượng ảnh đầu vào.

---
# III. CÁC CÂU HỎI MỞ RỘNG

## 3.1. Làm thế nào để đánh giá chất lượng của các cạnh được phát hiện bởi Canny?

### Phương pháp định lượng (Quantitative)

1. **So sánh với Ground Truth:**
   - Sử dụng bộ dữ liệu có nhãn (BSDS500).
   - **Precision** = TP / (TP + FP): Tỷ lệ cạnh đúng / tổng cạnh phát hiện.
   - **Recall** = TP / (TP + FN): Tỷ lệ cạnh đúng được phát hiện.
   - **F1-score** = 2 × Precision × Recall / (Precision + Recall).

2. **Pratt's Figure of Merit (FOM):**

$$FOM = \frac{1}{\max(N_{ideal}, N_{detected})} \sum_{i=1}^{N_{detected}} \frac{1}{1 + \alpha \cdot d_i^2}$$

   - $d_i$: khoảng cách từ pixel cạnh phát hiện đến cạnh lý tưởng gần nhất.
   - FOM ∈ [0, 1], càng gần 1 càng tốt.

### Phương pháp định tính (Qualitative)

3. **Ba tiêu chí tối ưu của Canny:**
   - **Good detection**: Ít bỏ sót cạnh thật, ít phát hiện cạnh giả.
   - **Good localization**: Cạnh phát hiện sát vị trí thật.
   - **Single response**: Mỗi cạnh thật chỉ phát hiện 1 lần.

4. **Đánh giá trực quan:** Cạnh liên tục? Mỏng? Ít nhiễu? Không bỏ sót?

## 3.2. Có những phương pháp nào khác để cải thiện hiệu suất của Canny?

### Cải thiện tiền xử lý

- **Bilateral Filter**: Giảm nhiễu nhưng giữ cạnh sắc nét hơn Gaussian.
- **Non-Local Means Denoising**: Hiệu quả cho nhiễu phức tạp.
- **CLAHE**: Tăng tương phản cục bộ trước khi phát hiện cạnh.

### Cải thiện xử lý chính

- **Tự động chọn ngưỡng:** Phương pháp Otsu hoặc Median-based.
  - $T_{high} = 0.33 \times \text{median}(I)$, $T_{low} = 0.66 \times \text{median}(I)$.
- **Scharr operator**: Thay Sobel, chính xác hơn cho tính gradient.
- **Adaptive Canny**: Ngưỡng thay đổi cục bộ theo vùng ảnh.

### Cải thiện hậu xử lý

- **Morphological operations**: Dilation/Erosion để nối cạnh đứt đoạn.
- **Skeletonization**: Làm mỏng cạnh thêm.

### Deep Learning

- **HED (Holistically-Nested Edge Detection)**: Deep Learning, tốt hơn Canny trên nhiều benchmark.
- **RCF (Richer Convolutional Features)**: Cải tiến từ HED.

## 3.3. Canny có thể phát hiện cạnh trong ảnh màu không?

**Có.** Có nhiều cách tiếp cận:

| Cách | Phương pháp | Ưu điểm | Nhược điểm |
|:---|:---|:---|:---|
| **1** | Chuyển Grayscale → Canny | Đơn giản, nhanh | Mất thông tin màu |
| **2** | Canny trên từng kênh R, G, B → OR | Nhiều cạnh hơn | Có thể nhiều cạnh giả |
| **3** | Canny trên kênh L (Lab) hoặc V (HSV) | Tách tốt sáng/màu | Phức tạp hơn |
| **4** | Di Zenzo multi-channel gradient | Chính xác nhất | Tốn tính toán |

**Chi tiết:**

- **Cách 1** (phổ biến nhất): Chuyển RGB → Grayscale bằng công thức weighted average, rồi áp dụng Canny bình thường. Nhược điểm là mất thông tin khi 2 vùng khác màu nhưng cùng độ sáng.

- **Cách 2**: Tách 3 kênh R, G, B, áp dụng Canny riêng biệt, sau đó kết hợp bằng phép OR (hợp). Phát hiện được nhiều cạnh hơn nhờ thông tin đa kênh.

- **Cách 3**: Chuyển sang Lab/HSV, áp dụng Canny trên kênh luminance (L) kết hợp kênh chrominance (a, b). Tách biệt tốt hơn giữa độ sáng và sắc độ.

- **Cách 4**: Tính Di Zenzo gradient tensor đa kênh, lấy eigenvalue lớn nhất. Chính xác nhất nhưng phức tạp.

→ Phần minh họa bằng code xem ở file `Phan2_ThucHanh_Canny.ipynb`.

## 3.4. Làm thế nào để áp dụng Canny cho video?

### Nguyên tắc

Áp dụng Canny cho video = áp dụng Canny cho **từng frame** của video.

### Quy trình

1. Đọc video bằng `cv2.VideoCapture()`.
2. Lặp qua từng frame:
   - Chuyển frame → grayscale.
   - Áp dụng Gaussian Blur.
   - Áp dụng Canny.
3. Hiển thị hoặc lưu kết quả.

### Mã giả (Pseudocode)

```python
cap = cv2.VideoCapture('video.mp4')  # hoặc 0 cho webcam

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 1.4)
    edges = cv2.Canny(blur, 50, 150)
    
    cv2.imshow('Canny Video', edges)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
```

### Kỹ thuật tối ưu

- **Temporal smoothing**: Trung bình cạnh qua nhiều frame → giảm nhấp nháy.
- **Adaptive thresholding**: Tự động thay đổi ngưỡng theo ánh sáng từng frame.
- **Background subtraction + Canny**: Chỉ phát hiện cạnh vùng chuyển động.
- **GPU acceleration**: Dùng `cv2.cuda` cho xử lý thời gian thực.

### Ứng dụng

- **Lane detection**: Phát hiện làn đường cho xe tự lái.
- **Video surveillance**: Giám sát an ninh, phát hiện xâm nhập.
- **Augmented Reality**: Phát hiện cạnh đối tượng để overlay thông tin.
- **Motion edge detection**: Phát hiện chuyển động qua thay đổi cạnh.

→ Phần minh họa bằng code xem ở file `Phan2_ThucHanh_Canny.ipynb`.

---

# TỔNG KẾT

| # | Nội dung | Điểm chính |
|:--|:---|:---|
| 1 | Thuật toán Canny | 5 bước: Gaussian → Gradient → NMS → Double Threshold → Edge Tracking |
| 2 | Tham số | $\sigma$ (mức làm mịn), $T_{low}$ (ngưỡng thấp), $T_{high}$ (ngưỡng cao) |
| 3 | So sánh | Canny **chính xác nhất**, nhưng **chậm hơn** Sobel/Laplacian |
| 4 | Đánh giá | Precision, Recall, F1-score, Pratt's FOM |
| 5 | Cải thiện | Bilateral Filter, Auto-threshold, CLAHE, Deep Learning (HED) |
| 6 | Ảnh màu | 4 cách: Grayscale, RGB channels, Lab space, Di Zenzo gradient |
| 7 | Video | Áp dụng Canny cho từng frame + temporal smoothing |